In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 70.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 79.6 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 bitsandbytes==0.46.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 25.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.1 MB/s eta 0:00:00:00:01


In [3]:
LORA_PATH = "/kaggle/input/datasets/xyz2005/lora-output"

In [4]:
import sys

print(sys.executable)

/usr/bin/python3


In [5]:
!{sys.executable} -m pip install -q streamlit pyngrok

In [6]:
import streamlit

print("Streamlit version:", streamlit.__version__)
print("Streamlit location:", streamlit.__file__)

Streamlit version: 1.61.1
Streamlit location: /usr/local/lib/python3.12/dist-packages/streamlit/__init__.py


In [7]:
import subprocess
import sys
import time

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "/kaggle/working/app.py",
        "--server.port=8501",
        "--server.address=127.0.0.1",
        "--server.headless=true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("PID:", process.pid)
print("Status:", process.poll())

PID: 130
Status: 2


In [25]:
%%writefile /kaggle/working/app.py

import streamlit as st
from PIL import Image
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

LORA_PATH = "/kaggle/input/datasets/xyz2005/lora-output"

# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="ConceptAlign-Rad",
    page_icon="🩻",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# BLUE THEME
# ============================================================

st.markdown(
    """
    <style>

    .stApp {
        background: linear-gradient(
            180deg,
            #F4F9FF 0%,
            #E8F2FC 100%
        );
    }

    section[data-testid="stSidebar"] {
        background: linear-gradient(
            180deg,
            #0D47A1 0%,
            #1565C0 100%
        );
    }

    section[data-testid="stSidebar"] * {
        color: #FFFFFF !important;
    }

    section[data-testid="stSidebar"] label {
        font-weight: 500;
    }

    h1, h2, h3 {
        color: #0D47A1 !important;
        font-weight: 700 !important;
    }

    p, .stMarkdown, .stCaption {
        color: #1A2E44;
    }

    .stButton > button {
        background: linear-gradient(
            90deg,
            #1565C0,
            #1E88E5
        );

        color: #FFFFFF;
        border: none;
        border-radius: 8px;
        padding: 0.6rem 1.4rem;
        font-weight: 600;
    }

    .stButton > button:hover {
        background: linear-gradient(
            90deg,
            #0D47A1,
            #1565C0
        );

        transform: translateY(-1px);
        box-shadow: 0 4px 10px rgba(
            13,
            71,
            161,
            0.3
        );
    }

    div[data-testid="stMetric"] {
        background-color: #FFFFFF;
        border: 1px solid #BBDEFB;
        border-radius: 12px;
        padding: 1rem;
        box-shadow: 0 2px 6px rgba(
            21,
            101,
            192,
            0.08
        );
    }

    div[data-testid="stFileUploader"] {
        background-color: #FFFFFF;
        border: 1px dashed #64B5F6;
        border-radius: 12px;
        padding: 0.8rem;
    }

    .stTextArea textarea {
        border: 1px solid #90CAF9;
        border-radius: 8px;
    }

    div[data-testid="stVerticalBlockBorderWrapper"] {
        background-color: #FFFFFF;
        border: 1px solid #D6E9FB;
        border-radius: 14px;
        padding: 0.5rem;
        box-shadow: 0 2px 8px rgba(
            21,
            101,
            192,
            0.06
        );
    }

    hr {
        border-top: 1px solid #BBDEFB;
    }

    </style>
    """,
    unsafe_allow_html=True
)

# ============================================================
# TITLE
# ============================================================

st.title("🩻 ConceptAlign-Rad")

st.subheader(
    "Concept-Aligned AI System for Automated Radiology Report Generation"
)

st.write(
    "Upload a chest X-ray, select the detected medical concepts, "
    "and generate a structured radiology report using the "
    "LoRA fine-tuned Qwen language model."
)

st.info(
    "📊 To view evaluation results, select **Model Metrics** "
    "from the sidebar."
)

# ============================================================
# LOAD MODEL
# ============================================================

@st.cache_resource
def load_model():

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=(
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
        device_map=(
            "auto"
            if torch.cuda.is_available()
            else None
        )
    )

    model = PeftModel.from_pretrained(
        base_model,
        LORA_PATH
    )

    model.eval()

    return tokenizer, model


with st.spinner("Loading ConceptAlign-Rad model..."):

    tokenizer, model = load_model()

st.success("Model loaded successfully!")

# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header("🧠 Medical Concepts")

st.sidebar.caption(
    "Select the concepts detected in the X-ray."
)

LABELS = [
    "Pneumonia",
    "Cardiomegaly",
    "Pleural Effusion",
    "Atelectasis",
    "Edema",
    "Pneumothorax",
    "Consolidation",
    "Lung Opacity",
    "Nodule",
    "Mass"
]

selected_concepts = []

for label in LABELS:

    if st.sidebar.checkbox(label):

        selected_concepts.append(label)

# ============================================================
# IMAGE UPLOAD
# ============================================================

st.header("1. Upload Chest X-Ray")

uploaded_file = st.file_uploader(
    "Choose a chest X-ray image",
    type=[
        "jpg",
        "jpeg",
        "png"
    ]
)

if uploaded_file is not None:

    image = Image.open(
        uploaded_file
    )

    col1, col2 = st.columns(2)

    with col1:

        with st.container(border=True):

            st.subheader("Chest X-Ray")

            st.image(
                image,
                caption="Uploaded Chest X-Ray",
                use_container_width=True
            )

    with col2:

        with st.container(border=True):

            st.subheader(
                "Detected / Selected Concepts"
            )

            if selected_concepts:

                for concept in selected_concepts:

                    st.success(
                        "✓ " + concept
                    )

            else:

                st.info(
                    "Select the medical concepts "
                    "from the sidebar."
                )

# ============================================================
# REPORT GENERATION
# ============================================================

st.header("2. Generate Radiology Report")

if st.button(
    "📝 Generate Report",
    type="primary"
):

    if uploaded_file is None:

        st.warning(
            "Please upload a chest X-ray first."
        )

    elif len(selected_concepts) == 0:

        st.warning(
            "Please select at least one "
            "medical concept."
        )

    else:

        findings = "\n".join(
            [
                f"- {x}"
                for x in selected_concepts
            ]
        )

        prompt = f"""### System
You are an expert radiologist specializing
in chest X-ray interpretation.

### User

Generate a professional radiology report
from the following findings.

Findings:
{findings}

### Assistant
"""

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        )

        if torch.cuda.is_available():

            inputs = {
                k: v.to(model.device)
                for k, v in inputs.items()
            }

        with st.spinner(
            "Generating radiology report..."
        ):

            with torch.no_grad():

                output = model.generate(
                    **inputs,
                    max_new_tokens=120,
                    do_sample=False,
                    repetition_penalty=1.15,
                    pad_token_id=(
                        tokenizer.eos_token_id
                    )
                )

        generated_text = tokenizer.decode(
            output[0],
            skip_special_tokens=True
        )

        # Extract assistant response
        if "### Assistant" in generated_text:

            report = generated_text.split(
                "### Assistant"
            )[-1].strip()

        else:

            report = generated_text

        report = report.replace(
            "### Assistant",
            ""
        ).strip()

        # ====================================================
        # DISPLAY REPORT
        # ====================================================

        st.subheader(
            "3. Generated Radiology Report"
        )

        with st.container(border=True):

            st.text_area(
                "Report",
                report,
                height=250
            )

# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "ConceptAlign-Rad · Base model Qwen2.5-1.5B "
    "fine-tuned with LoRA · Prototype Implementation"
)

Overwriting /kaggle/working/app.py


In [26]:
%%writefile /kaggle/working/pages/2_Model_Metrics.py

import streamlit as st

# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="ConceptAlign-Rad | Metrics",
    page_icon="📊",
    layout="wide"
)

# ============================================================
# BLUE THEME
# ============================================================

st.markdown(
    """
    <style>

    .stApp {
        background: linear-gradient(
            180deg,
            #F4F9FF 0%,
            #E8F2FC 100%
        );
    }

    section[data-testid="stSidebar"] {
        background: linear-gradient(
            180deg,
            #0D47A1 0%,
            #1565C0 100%
        );
    }

    section[data-testid="stSidebar"] * {
        color: #FFFFFF !important;
    }

    h1, h2, h3 {
        color: #0D47A1 !important;
        font-weight: 700 !important;
    }

    div[data-testid="stMetric"] {
        background-color: #FFFFFF;
        border: 1px solid #BBDEFB;
        border-radius: 12px;
        padding: 1rem;
    }

    </style>
    """,
    unsafe_allow_html=True
)

# ============================================================
# TITLE
# ============================================================

st.title("📊 Model Performance Metrics")

st.write(
    "Prototype evaluation results for "
    "ConceptAlign-Rad."
)

st.caption(
    "Evaluation performed on 496 generated reports."
)

# ============================================================
# MODEL INFORMATION
# ============================================================

st.header("Model Information")

col1, col2, col3 = st.columns(3)

with col1:

    st.metric(
        "Base Model",
        "Qwen2.5-1.5B"
    )

with col2:

    st.metric(
        "Fine-Tuning",
        "LoRA"
    )

with col3:

    st.metric(
        "Task",
        "Radiology Report Generation"
    )

# ============================================================
# EVALUATION RESULTS
# ============================================================

st.divider()

st.header("Prototype Evaluation Results")

# ------------------------------------------------------------
# Main metrics
# ------------------------------------------------------------

col1, col2, col3, col4 = st.columns(4)

with col1:

    st.metric(
        "BERTScore F1",
        "0.8351"
    )

with col2:

    st.metric(
        "CheXbert F1",
        "0.5504"
    )

with col3:

    st.metric(
        "RadGraph Entity F1",
        "0.0744"
    )

with col4:

    st.metric(
        "BLEU",
        "0.0055"
    )

# ============================================================
# ROUGE
# ============================================================

st.subheader("ROUGE Metrics")

col1, col2, col3 = st.columns(3)

with col1:

    st.metric(
        "ROUGE-1",
        "0.1201"
    )

with col2:

    st.metric(
        "ROUGE-2",
        "0.0145"
    )

with col3:

    st.metric(
        "ROUGE-L",
        "0.0891"
    )

# ============================================================
# RADGRAPH
# ============================================================

st.subheader("RadGraph Metrics")

col1, col2, col3 = st.columns(3)

with col1:

    st.metric(
        "Entity F1",
        "0.0744"
    )

with col2:

    st.metric(
        "Entity + Relation F1",
        "0.0654"
    )

with col3:

    st.metric(
        "Combined",
        "0.0287"
    )

# ============================================================
# GENERATION STATISTICS
# ============================================================

st.divider()

st.header("Generation Statistics")

col1, col2, col3 = st.columns(3)

with col1:

    st.metric(
        "Avg. Generated Length",
        "73.17 words"
    )

with col2:

    st.metric(
        "Avg. Reference Length",
        "40.08 words"
    )

with col3:

    st.metric(
        "Repetition Ratio",
        "0.074"
    )

# ============================================================
# INTERPRETATION
# ============================================================

st.divider()

st.header("Initial Observations")

st.info(
    """
    **BERTScore F1 = 0.8351** indicates relatively strong
    semantic similarity between generated and reference reports.

    **CheXbert F1 = 0.5504** indicates moderate consistency
    in clinically relevant findings.

    The lower BLEU, ROUGE and RadGraph scores indicate that
    the current prototype still requires improvement in
    exact terminology, clinical entities and relationships.
    """
)

st.caption(
    "Note: These values represent prototype evaluation results "
    "and should not be interpreted as clinical diagnostic accuracy."
)

Writing /kaggle/working/pages/2_Model_Metrics.py


In [27]:
!fuser -k 8501/tcp


8501/tcp:              321


In [28]:
import subprocess
import sys
import time

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "/kaggle/working/app.py",
        "--server.port=8501",
        "--server.address=127.0.0.1",
        "--server.headless=true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)
status = process.poll()
print("PID:", process.pid)
print("Status:", status)

# If it crashed, read and print the error logs
if status is not None and status != 0:
    stdout, _ = process.communicate()
    print("\n--- STREAMLIT ERROR LOGS ---")
    print(stdout)


PID: 390
Status: None


In [23]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8501",
        timeout=10
    )

    print("Streamlit status:", response.status_code)

except Exception as e:
    print("ERROR:", e)

Streamlit status: 200


In [ ]:
import os

print(os.listdir(LORA_PATH))

In [29]:
from pyngrok import ngrok

ngrok.kill()

ngrok.set_auth_token("3HZsrQtqHQ6cpI5TXWzPWPNbw68_4eTiwSFReDzLGx8hMCoFt")

public_url = ngrok.connect(8501)

print("ConceptAlign-Rad UI:")
print(public_url)

ConceptAlign-Rad UI:
NgrokTunnel: "https://giddy-crunchy-primp.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
import os

print(os.path.exists("app.py"))

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8501",
    timeout=10
)

print("Streamlit status:", response.status_code)